In [ ]:
import tifffile as tiff
import numpy as np
from spatialdata import SpatialData
from spatialdata.models import Image2DModel, PointsModel
import harpy as hp
import pandas as pd
from pathlib import Path
import spatialdata_plot
from napari_spatialdata import Interactive

import torch
from cellpose import models
from harpy.image import cellpose_callable

In [ ]:
unit_testing = False 

# Different sections:
### 1. Reading in the data
### 2. Image processing
### 3. Cell segmentation
### 4. Transcript allocation
### 5. Single-cell analysis: 
##### 5.1 filtering, normalization and qc
##### 5.2 clustering
##### 5.3 cell type annotation
##### 5.4 

## Reading in the data

In [ ]:
file_path = "/Volumes/Intenso/spatial-transcriptomics/"

In [ ]:
# loading the image and parsing it 
img = tiff.imread(f"{file_path}A1_DAPI.tiff")  # shape likely (y, x) or (c, y, x)
if img.ndim == 2:
    img = img[None, ...]

if unit_testing == True:
    x_min, x_max, y_min, y_max = 6000, 14192, 6000, 14192
    img = img[:, y_min:y_max, x_min:x_max]

image = Image2DModel.parse(img, dims=("c", "y", "x"))

In [ ]:
# loading the transcripts data
df = pd.read_csv(
    f"{file_path}A1_results.txt",
    sep=r"\s+",
    header=None,
    names=["x", "y", "z", "gene"],
    engine="python",
)
df.head(10)

In [ ]:
points = PointsModel.parse(df, coordinates={"x": "x", "y": "y"})

In [ ]:
# creating the sdata object
sdata = SpatialData(
    images={"a1_dapi": image},
    points={"a1_genes": points},
)
sdata

In [ ]:
sdata.pl.render_images("a1_dapi", cmap = "gray").pl.show()

In [ ]:
hp.pl.plot_image(
    sdata, 
    img_layer = "a1_dapi" , 
    crd = [6000, 11000, 6000, 11000], # (xmin, xmax, ymin, ymax). 
    figsize = (10,10),
)

In [ ]:
sdata["a1_genes"].compute()

In [ ]:
# write the object to zarr
# #sdata.write("/Volumes/Intenso/spatial-transcriptomics/intermediate_results/20251223.zarr", overwrite=True)

## Image processing

In [ ]:
sdata = hp.im.min_max_filtering(
    sdata,
    img_layer = "a1_dapi",
    output_layer = "a1_min_max_filtered",
    size_min_max_filter = 51,
    overwrite = True,
)

hp.pl.plot_image(
    sdata, 
    img_layer=[ "a1_dapi", "a1_min_max_filtered" ], 
    crd = [4000, 8000, 6000, 8000], 
    figsize=(20,20)
)

In [ ]:
sdata = hp.im.enhance_contrast(
    sdata,
    img_layer = "a1_min_max_filtered",
    output_layer = "a1_clahe",
    contrast_clip = 20,
    chunks = 20000,
    overwrite = True
)

# Plot the contrast enhanced image
hp.pl.plot_image(
    sdata, 
    img_layer=[ "a1_min_max_filtered", "a1_clahe" ], 
    crd = [4000, 8000, 6000, 8000], 
    figsize=(20,20)
)

In [ ]:
sdata

## Segmentation

In [ ]:
# # rechuncking on disk
# from spatialdata.transformations import get_transformation

# sdata=hp.im.add_image_layer(
#     sdata,
#     arr=sdata["a1_clahe"].data.rechunk( 2048 ),
#     transformations=get_transformation( sdata["a1_clahe"], get_all=True ),
#     output_layer = "a1_clahe",
#     overwrite=True,
# )

In [ ]:
sdata["a1_clahe"].data.chunks

In [ ]:
from packaging import version
import cellpose
import torch

cellpose_version = version.parse(cellpose.version)
if torch.backends.mps.is_available() and cellpose_version >= version.parse("4.0"):  # mps bugged in cellpose < 4.0
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}.")

In [ ]:
from dask.distributed import Client, LocalCluster

# # Create a local Dask cluster
cluster = LocalCluster(
     n_workers=1,              # Number of worker processes
     threads_per_worker=3,    # Number of threads per worker
     processes=False,
     memory_limit="12GB",      # Memory limit per worker
 )

# # Connect a Client to the cluster
client = Client(cluster)

# # Print the Dask dashboard link
print(client.dashboard_link)

In [ ]:
# Perform nucleus segmentation
sdata = hp.im.segment(
    sdata,
    img_layer="a1_clahe", # The image layer in sdata to be segmented.
    chunks=4096, #settings chunks=None would be equivalent to settings chunks=2048, as chunks on disk are 2048
    depth=40,
    model=cellpose_callable,
    # parameters that will be passed to the callable _cellpose:
    pretrained_model="nuclei", # can also be "cyto", "cyto3", or a path to a fine-tuned cellpose model.
    device=device,
    diameter=80,
    flow_threshold=0.6,
    cellprob_threshold=-6,
    min_size=40,
    output_labels_layer="a1_segmentation_mask",
    output_shapes_layer="a1_segmentation_mask_boundaries",
    #crd=[6000, 10096, 6000, 10096] if unit_testing else None,  # region to segment [x_min, xmax, y_min, y_max],
    overwrite=True,
)


In [ ]:
client.close()
cluster.close()

In [ ]:
hp.pl.plot_image(
    sdata, 
    img_layer="a1_clahe", 
    crd = [2000, 4000, 2000, 4000], 
    figsize=(10,10)
)

In [ ]:
import spatialdata_plot
hp.pl.plot_shapes(
    sdata, 
    img_layer="a1_clahe", 
    shapes_layer="a1_segmentation_mask_boundaries", 
    figsize=(10,10), 
    crd = [2000, 4000, 2000, 4000]
)

In [ ]:
sdata

In [ ]:
# write the object to zarr
sdata.write("/Volumes/Intenso/spatial-transcriptomics/intermediate_results/20251230_v2.zarr", overwrite=True)

In [ ]:
import spatialdata as sd
from_disk = sd.read_zarr("/Volumes/Intenso/spatial-transcriptomics/intermediate_results/20251230_v2.zarr",
                         on_bad_files="warn",)

In [ ]:
# before loading from zarr, get rid of the annoying ""._*" file
!find "/Volumes/Intenso/spatial-transcriptomics/intermediate_results/20251230_v2.zarr" -name "._*"

In [ ]:
sdata = from_disk
sdata

## Transcript allocation

In [ ]:
sdata = hp.tb.allocate(
    sdata=sdata,
    labels_layer="a1_segmentation_mask", # The labels layer (i.e. segmentation mask) in `sdata` to be used to allocate the transcripts to cells.
    points_layer="a1_genes", # The points layer in `sdata` that contains the transcripts.
    output_layer="a1_transcriptomics", # The table layer in `sdata` in which to save the AnnData object with the transcripts counts per cell.
    update_shapes_layers=False,
    overwrite=True,
)

In [ ]:
sdata

In [ ]:
sdata.points["a1_genes"].head()

In [ ]:
sdata.tables["a1_transcriptomics"].to_df().head()

In [ ]:
hp.pl.plot_shapes(
    sdata,
    img_layer="a1_clahe",
    shapes_layer="a1_segmentation_mask_boundaries",
    figsize=(5,5),
    crd=[6000, 10000, 2500, 6500],
    table_layer="a1_transcriptomics",
    column="Aldh1l1",
)

In [ ]:
sdata = hp.im.transcript_density(
    sdata,
    img_layer="a1_clahe", # The layer of the SpatialData object used for determining image boundary.
    points_layer="a1_genes", # The layer name that contains the transcript data points, by default "transcripts".
    output_layer="a1_gene_density", # The name of the output image layer
    overwrite=True,
)

In [ ]:
hp.pl.plot_image(
    sdata, 
    img_layer = ["a1_clahe", "a1_gene_density"], 
    figsize=(10,10)
)

In [ ]:
# check the percentage of transcript allocated to segmented cells
print('Number of transcripts in points layer: ', len(sdata.points["a1_genes"]))
print('Number of transcripts assigned to cells: ', sdata.tables["a1_transcriptomics"].X.sum())
print('Percentage of transcripts allocated: ', ((sdata.tables["a1_transcriptomics"].X.sum())/len(sdata.points["a1_genes"]))*100)

In [ ]:
# check the nr of genes
print('Number of genes in points layer: ', sdata.points['a1_genes'].compute()['gene'].nunique())
print('Number of genes found in cells: ', len(sdata.tables["a1_transcriptomics"].var.index))


In [ ]:

df_analyse_genes_left_out = hp.pl.analyse_genes_left_out(
    sdata,
    labels_layer="a1_segmentation_mask",
    table_layer="a1_transcriptomics",
    points_layer="a1_genes",
)

In [ ]:
df_analyse_genes_left_out.sort_values(by="proportion_kept", ascending=True)

## Single-cell analysis

### Filtering & normalization

In [ ]:
sdata = hp.tb.preprocess_transcriptomics(
    sdata,
    labels_layer="a1_segmentation_mask",
    table_layer="a1_transcriptomics",
    output_layer="a1_transcriptomics_preprocessed", # write results to a new slot, we could also write to the same slot (when passing overwrite==True).
    min_counts=10,
    min_cells=5,
    size_norm=True,
    highly_variable_genes=False,  # If True, will only retain highly variable genes. This can be used for transcriptome-wide methods.
    max_value_scale=10, # The maximum value to which data will be scaled
    n_comps=50, # Number of principal components to calculate.
    overwrite=True,
    update_shapes_layers=False,
)

In [ ]:
sdata.tables[ "a1_transcriptomics_preprocessed" ]

In [ ]:
hp.pl.preprocess_transcriptomics(
    sdata,
    table_layer="a1_transcriptomics_preprocessed",
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(sdata.tables["a1_transcriptomics_preprocessed"].obs["shapeSize"], kde=False)
plt.title("Area of Segmentation Masks")
plt.xlabel("shapeSize (pixels)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
hp.pl.plot_shapes(
    sdata,
    img_layer="a1_clahe",
    table_layer="a1_transcriptomics_preprocessed",
    column="total_counts",
    shapes_layer="a1_segmentation_mask_boundaries",
    crd=[6000, 10000, 2500, 6500],
    figsize=(8,8)
)

In [ ]:
sdata

In [ ]:
# Filter cells on size
sdata = hp.tb.filter_on_size(
    sdata,
    labels_layer="a1_segmentation_mask",
    table_layer="a1_transcriptomics_preprocessed",
    output_layer="transcriptomics_filter",
    min_size=500, # Minimum cell size
    max_size=100000, # Maximum cell size
    update_shapes_layers=True,
    overwrite=True,
)

In [ ]:
hp.pl.plot_shapes(
    sdata, 
    img_layer="a1_clahe", 
    shapes_layer="a1_segmentation_mask_boundaries", 
    shapes_layer_filtered="filtered_size_a1_segmentation_mask_boundaries", # Filtered cells will be plotted in red.
    figsize=(5,5), 
    crd = [6000, 10000, 2500, 6500]
)

### Clustering

In [ ]:
import scanpy as sc

# Leiden clustering
sdata = hp.tb.leiden(
    sdata,
    labels_layer="a1_segmentation_mask",
    table_layer="a1_transcriptomics_filter",
    output_layer="a1_transcriptomics_clustered_res0.95",
    calculate_umap=True,
    calculate_neighbors=True,
    n_pcs=17, # The number of principal components to use when calculating neighbors.
    n_neighbors=35, # The number of neighbors to consider when calculating neighbors.
    resolution=0.95,
    rank_genes=True,
    key_added="leiden",
    overwrite=True,
)

# Plot UMAP
sc.pl.umap(sdata.tables["a1_transcriptomics_clustered_res1"], color=["leiden"], show=True)

In [ ]:
hp.pl.plot_shapes(
    sdata,
    img_layer="a1_clahe",
    table_layer="a1_transcriptomics_clustered_res0.95",
    column="leiden",
    shapes_layer="a1_segmentation_mask_boundaries",
    alpha=1.0,
    linewidth=0,
    cmap='tab20'
    # crd=[2000, 4000, 2000, 4000]
)

In [ ]:
sc.pl.rank_genes_groups(
    sdata.tables["a1_transcriptomics_clustered_res0.95"], 
    n_genes=12, 
    sharey=False, 
    show=True)

In [ ]:
cluster_to_type = {
    "0": "Dentate Gyrus Granule Cell Layer",
    "1": "Astrocytes Protoplasmic",
    "2": "Endothelial cells",
    "3": "Oligodendrocytes",
    "4": "Cortical Neurons",
    "5": "Thalamic Neurons",
    "6": "CA2 and CA3 Neurons",
    "7": "Oligodendrocytes_2",
    "8": "Medial Habenula Neurons",
    "9": "CA1 Neurons",
    "10": "Dentate Gyrus Subgranular Layer",
    "11": "OPCs",
    "12": "Inhibitory Neurons",
    "13": "Choriod Plexus Cells",
    "14": "Astrocytes Fibrous",
    "15": "Microglia",
}

In [ ]:
marker_genes_dict = {
    "Dentate Gyrus Granule Cell Layer": ["Olfm1", "Plxna4"],
    "Astrocytes Protoplasmic": ["Aldoc", "Slc1a2"],
    "Endothelial cells": ["Pecam1", "Flt1"],
    "Oligodendrocytes": ["Mog", "Plp1"],
    "Cortical Neurons": ["Ptprd", "Plxna2"],
    "Thalamic Neurons": ["L1cam", "Tenm4"],
    "CA2 and CA3 Neurons": ["Brinp2", "Adam11"],
    "Medial Habenula Neurons": ["Alcam", "Nrp2"],
    "CA1 Neurons": ["Epha6", "Nrxn3"],
    "Choriod Plexus Cells": ["Tenm4", "Lrp1"],
    "Dentate Gyrus Subgranular Layer": ["Ptprs", "Tubb3"],
    "OPCs": ["Tnr", "Vcan"],
    "Inhibitory Neurons": ["Gad1", "Gad2"],
    "Astrocytes Fibrous": ["Aqp4", "Hepacam"],
    "Microglia": ["Csf1r", "C1qa"]
}

In [ ]:
list(sdata.tables.keys())

In [ ]:
table_key = list(sdata.tables.keys())[12]
adata = sdata.tables[table_key]

In [ ]:
adata.obs["leiden_str"] = adata.obs["leiden"].astype(str)

In [ ]:
adata.obs["cell_type"] = adata.obs["leiden_str"].map(cluster_to_type).astype("category")

In [ ]:
adata

In [ ]:
sdata.tables["a1_transcriptomics_clustered_res0.95"].obs["cell_type"]

In [ ]:
sc.pl.dotplot(
    sdata.tables["a1_transcriptomics_clustered_res0.95"], 
    var_names=marker_genes_dict, 
    groupby="leiden", 
    cmap="Blues",
)

In [ ]:
sc.pl.umap(sdata.tables["a1_transcriptomics_clustered_res0.95"], color=["cell_type"])

In [ ]:
hp.pl.plot_shapes(
    sdata,
    column="cell_type",
    img_layer="a1_clahe",
    table_layer= "a1_transcriptomics_clustered_res0.95",
    shapes_layer="a1_segmentation_mask_boundaries",
    linewidth=0,
    alpha=0.7,
    crd=None,
    cmap="tab20"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

table_layer = "a1_transcriptomics_clustered_res0.95"
obs = sdata.tables[table_layer].obs

cell_types = obs["cell_type"].dropna().unique()

for ct in sorted(cell_types):
    # create temporary masked column
    temp_col = f"cell_type_{ct}"
    obs[temp_col] = np.where(obs["cell_type"] == ct, obs["cell_type"], np.nan)

    plt.figure(figsize=(8, 8))
    hp.pl.plot_shapes(
        sdata,
        column=temp_col,
        img_layer="a1_clahe",
        table_layer=table_layer,
        shapes_layer="a1_segmentation_mask_boundaries",
        linewidth=0,
        alpha=0.7,
        crd=None,
        cmap="tab20"
    )
    plt.title(ct)
    plt.show()

# optional cleanup: remove temp columns afterwards
obs.drop(columns=[c for c in obs.columns if c.startswith("cell_type_")], inplace=True)

